<a href="https://colab.research.google.com/github/Dhanalakshmi34/Dhanam/blob/main/JSON_Refiner_Advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio json5 jsonschema python-dotenv pydantic


In [ ]:
def normalize_key(key, style):

    if style == CaseStyle.SNAKE_CASE:
        return to_snake_case(key)

    elif style == CaseStyle.CAMEL_CASE:
        return to_camel_case(key)

    elif style == CaseStyle.KEBAB_CASE:
        return to_kebab_case(key)

    elif style == CaseStyle.PASCAL_CASE:
        return to_pascal_case(key)

    return key


In [ ]:
def parse_key_value(text):

    result = {}

    lines = text.split("\n")

    for line in lines:
        if ":" in line:
            key, value = line.split(":",1)
            result[key.strip()] = infer_type(value.strip())

    return result


In [ ]:
def validate_json(data, schema):

    try:
        validate(instance=data, schema=schema)
        return "✅ JSON is valid"

    except ValidationError as e:
        return f"❌ Validation Error: {e.message}"


In [ ]:
def flatten_json(data, parent_key="", sep="."):

    items = {}

    for k,v in data.items():

        new_key = parent_key + sep + k if parent_key else k

        if isinstance(v, dict):
            items.update(flatten_json(v,new_key,sep))

        else:
            items[new_key] = v

    return items


In [ ]:
def remove_nulls(data):

    return {k:v for k,v in data.items() if v is not None}


In [ ]:
def refine_json(text):

    data = parse_key_value(text)

    cleaned = remove_nulls(data)

    return json.dumps(cleaned,indent=4)


In [ ]:
with gr.Blocks() as demo:

    gr.Markdown("# JSON Refiner Advanced")

    input_text = gr.Textbox(lines=10,label="Enter Key Value Pairs")

    output_json = gr.Code(label="Generated JSON")

    btn = gr.Button("Convert")

    btn.click(refine_json,input_text,output_json)

demo.launch()


In [ ]:
import gradio as gr

with gr.Blocks() as demo:

    # ============= TAB 1: Core Refining =============
    with gr.TabItem("🔧 Core Refining"):

        with gr.Row():

            with gr.Column(scale=1):

                gr.Markdown("### 📝 Input Format")

                text_input = gr.Textbox(
                    label="Key-Value Text",
                    lines=12,
                    value="name: John Doe\nage: 28\nis_active: true\nbalance: 1500.50\nemail: john@example.com\ntags: [\"admin\", \"user\"]\nmetadata: {\"level\": 5}"
                )

                case_style = gr.Radio(
                    ["snake_case", "camelCase", "kebab-case", "PascalCase"],
                    value="snake_case",
                    label="🎨 Key Case Style"
                )

                with gr.Row():

                    remove_nulls = gr.Checkbox(
                        label="Remove Nulls",
                        value=False
                    )

                    flatten = gr.Checkbox(
                        label="Flatten",
                        value=False
                    )

                    pretty_print = gr.Checkbox(
                        label="Pretty Print",
                        value=True
                    )

demo.launch()

In [ ]:
import gradio as gr
from datetime import datetime

# Global history storage
transformation_history = []

def clear_history():
    """Clear transformation history"""
    global transformation_history
    transformation_history = []
    return "✅ History cleared!"

def download_history() -> str:
    """Download transformation history"""
    if not transformation_history:
        return "❌ No history to download"

    report = "JSON REFINER - TRANSFORMATION HISTORY\n"
    report += f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
    report += "=" * 80 + "\n\n"

    for i, item in enumerate(transformation_history, 1):
        report += f"Transformation #{i}\n"
        report += f"Timestamp: {item.get('timestamp')}\n"
        report += f"Input:\n{item.get('input')}\n\n"
        report += f"Output:\n{item.get('output')}\n\n"

        if item.get('errors'):
            report += f"Errors: {', '.join(item['errors'])}\n\n"

        report += "-" * 80 + "\n\n"

    return report


print("🚀 Loading JSON Refiner Advanced...")

with gr.Blocks(title="JSON Refiner - Advanced Edition") as demo:

    demo.load(None, js="""
    () => {
        document.body.style.background = 'linear-gradient(135deg, #0f172a 0%, #1e293b 100%)';

        const style = document.createElement('style');
        style.textContent = `
        :root {
            --primary-color: #667eea;
            --secondary-color: #f5576c;
        }

        body {
            background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%) !important;
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        }

        .gradio-container {
            background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%) !important;
            max-width: 1400px !important;
        }

        h1, h2, h3 {
            color: #f1f5f9 !important;
        }

        .code-block {
            background: #1e293b !important;
            border: 2px solid #667eea !important;
        }
        `;
        document.head.appendChild(style);
    }
    """)

    gr.Markdown("""
    # 🎯 JSON Refiner Advanced Edition
    ## Professional JSON Processing Tool

    **Features:**
    Type Inference • Schema Validation • Case Conversion • Flattening • Merging • History Tracking
    """)

    with gr.Tabs():
        with gr.Tab("History"):
            history_output = gr.Textbox(label="History Report", lines=20)

            download_btn = gr.Button("Download History")
            clear_btn = gr.Button("Clear History")

            download_btn.click(download_history, outputs=history_output)
            clear_btn.click(clear_history, outputs=history_output)

demo.launch()

In [ ]:
import json
from typing import Tuple

def merge_multiple_json(json_text1: str, json_text2: str, json_text3: str = "") -> Tuple[str, str]:
    """Merge multiple JSON objects"""

    try:
        objects = []

        for json_text in [json_text1, json_text2, json_text3]:
            if json_text.strip():
                obj = json.loads(json_text)
                objects.append(obj)

        if not objects:
            return "", "❌ No JSON objects to merge"

        # Merge JSON objects
        merged = {}
        for obj in objects:
            merged.update(obj)

        output = json.dumps(merged, indent=2, ensure_ascii=False)

        return output, f"✅ Merged {len(objects)} JSON objects successfully!"

    except json.JSONDecodeError as e:
        return "", f"❌ Invalid JSON format: {str(e)}"

    except Exception as e:
        return "", f"❌ Error: {str(e)}"

In [ ]:
import json
from typing import Tuple

def flatten_json_func(json_text: str) -> Tuple[str, str]:
    """Flatten JSON structure"""
    try:
        data = json.loads(json_text)
        flattened = flatten_json(data)

        output = json.dumps(flattened, indent=2, ensure_ascii=False)
        return output, "✅ JSON flattened!"

    except json.JSONDecodeError as e:
        return "", f"❌ Invalid JSON: {str(e)}"

    except Exception as e:
        return "", f"❌ Error: {str(e)}"


def unflatten_json_func(json_text: str) -> Tuple[str, str]:
    """Unflatten JSON structure"""
    try:
        data = json.loads(json_text)
        unflattened = unflatten_json(data)

        output = json.dumps(unflattened, indent=2, ensure_ascii=False)
        return output, "✅ JSON unflattened!"

    except json.JSONDecodeError as e:
        return "", f"❌ Invalid JSON: {str(e)}"

    except Exception as e:
        return "", f"❌ Error: {str(e)}"

In [ ]:
import gradio as gr
from datetime import datetime

# Global history
transformation_history = []

def clear_history():
    """Clear transformation history"""
    global transformation_history
    transformation_history = []
    return "✅ History cleared!"


def download_history() -> str:
    """Download transformation history"""

    if not transformation_history:
        return "❌ No history to download"

    report = "JSON REFINER - TRANSFORMATION HISTORY\n"
    report += f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
    report += "=" * 80 + "\n\n"

    for i, item in enumerate(transformation_history, 1):
        report += f"Transformation #{i}\n"
        report += f"Timestamp: {item['timestamp']}\n"
        report += f"Input:\n{item['input']}\n\n"
        report += f"Output:\n{item['output']}\n\n"

        if item.get("errors"):
            report += f"Errors: {', '.join(item['errors'])}\n\n"

        report += "-" * 80 + "\n\n"

    return report


print("🚀 Loading JSON Refiner Advanced...")

with gr.Blocks(title="JSON Refiner - Advanced Edition") as demo:

    demo.load(None, js="""
    () => {
        document.body.style.background = 'linear-gradient(135deg, #0f172a 0%, #1e293b 100%)';

        const style = document.createElement('style');

        style.textContent = `
        :root {
            --primary-color: #667eea;
            --secondary-color: #f5576c;
        }

        body {
            background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%) !important;
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        }

        .gradio-container {
            background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%) !important;
            max-width: 1400px !important;
        }

        h1, h2, h3 {
            color: #f1f5f9 !important;
        }

        .code-block {
            background: #1e293b !important;
            border: 2px solid #667eea !important;
        }
        `;

        document.head.appendChild(style);
    }
    """)

    gr.Markdown("""
    # 🎯 JSON Refiner Advanced Edition
    ## Professional JSON Processing Tool

    **Features:**
    Type Inference • Schema Validation • Case Conversion • Flattening • Merging • History Tracking
    """)

    with gr.Tabs():

        with gr.Tab("History"):

            history_output = gr.Textbox(
                label="History Report",
                lines=20
            )

            with gr.Row():
                download_btn = gr.Button("📥 Download History")
                clear_btn = gr.Button("🗑 Clear History")

            download_btn.click(
                fn=download_history,
                outputs=history_output
            )

            clear_btn.click(
                fn=clear_history,
                outputs=history_output
            )


demo.launch()

In [ ]:
import gradio as gr
from datetime import datetime

# Global history storage
transformation_history = []

def clear_history():
    """Clear transformation history"""
    global transformation_history
    transformation_history = []
    return "✅ History cleared!"


def download_history() -> str:
    """Download transformation history"""

    if not transformation_history:
        return "❌ No history to download"

    report = "JSON REFINER - TRANSFORMATION HISTORY\n"
    report += f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
    report += "=" * 80 + "\n\n"

    for i, item in enumerate(transformation_history, 1):
        report += f"Transformation #{i}\n"
        report += f"Timestamp: {item['timestamp']}\n"
        report += f"Input:\n{item['input']}\n\n"
        report += f"Output:\n{item['output']}\n\n"

        if item.get("errors"):
            report += f"Errors: {', '.join(item['errors'])}\n\n"

        report += "-" * 80 + "\n\n"

    return report


print("🚀 Loading JSON Refiner Advanced...")

with gr.Blocks(title="JSON Refiner - Advanced Edition") as demo:

    demo.load(
        None,
        js="""
        () => {
            document.body.style.background = 'linear-gradient(135deg, #0f172a 0%, #1e293b 100%)';

            const style = document.createElement('style');
            style.textContent = `
            :root {
                --primary-color: #667eea;
                --secondary-color: #f5576c;
            }

            body {
                background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%) !important;
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            }

            .gradio-container {
                background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%) !important;
                max-width: 1400px !important;
            }

            h1, h2, h3 { color: #f1f5f9 !important; }

            .code-block {
                background: #1e293b !important;
                border: 2px solid #667eea !important;
            }
            `;
            document.head.appendChild(style);
        }
        """
    )

    gr.Markdown("""
    # 🎯 JSON Refiner Advanced Edition
    ## Professional JSON Processing Tool

    **Features:**
    Type Inference • Schema Validation • Case Conversion • Flattening • Merging • History Tracking
    """)

    with gr.Tabs():

        with gr.Tab("History"):

            history_box = gr.Textbox(
                label="Transformation History",
                lines=20
            )

            with gr.Row():
                download_btn = gr.Button("📥 Download History")
                clear_btn = gr.Button("🗑 Clear History")

            download_btn.click(
                fn=download_history,
                outputs=history_box
            )

            clear_btn.click(
                fn=clear_history,
                outputs=history_box
            )

demo.launch()

In [ ]:
import gradio as gr

with gr.Blocks() as demo:

    with gr.Tabs():

        # ============= TAB 1: Core Refining =============
        with gr.Tab("🔧 Core Refining"):

            with gr.Row():

                with gr.Column(scale=1):

                    gr.Markdown("### 📝 Input Format")

                    text_input = gr.Textbox(
                        label="Key-Value Text",
                        lines=12,
                        value="""name: John Doe
age: 28
is_active: true
balance: 1500.50
email: john@example.com
tags: ["admin", "user"]
metadata: {"level": 5}"""
                    )

                    case_style = gr.Radio(
                        ["snake_case", "camelCase", "kebab-case", "PascalCase"],
                        value="snake_case",
                        label="🎨 Key Case Style"
                    )

                    with gr.Row():

                        remove_nulls = gr.Checkbox(
                            label="Remove Nulls",
                            value=False
                        )

                        flatten = gr.Checkbox(
                            label="Flatten",
                            value=False
                        )

                        pretty_print = gr.Checkbox(
                            label="Pretty Print",
                            value=True
                        )

demo.launch()

In [ ]:
import json
import re

def refine_json(text_input, case_style, remove_nulls, flatten, pretty_print):
    try:
        data = {}

        # Convert key:value text → dictionary
        lines = text_input.strip().split("\n")

        for line in lines:
            if ":" not in line:
                continue

            key, value = line.split(":", 1)
            key = key.strip()
            value = value.strip()

            # Basic type inference
            if value.lower() in ["true", "false"]:
                value = value.lower() == "true"
            elif re.match(r'^\d+$', value):
                value = int(value)
            elif re.match(r'^\d+\.\d+$', value):
                value = float(value)
            elif value.startswith("[") or value.startswith("{"):
                value = json.loads(value)

            data[key] = value

        # Remove nulls if enabled
        if remove_nulls:
            data = {k: v for k, v in data.items() if v is not None}

        # Pretty print option
        if pretty_print:
            output = json.dumps(data, indent=2)
        else:
            output = json.dumps(data)

        status = "✅ JSON refined successfully"
        stats = f"Keys: {len(data)}"

        return output, status, stats

    except Exception as e:
        return "", f"❌ Error: {str(e)}", ""

In [ ]:
!pip install -U gradio

In [ ]:
import gradio as gr
import json


# -----------------------------
# JSON Refiner Function
# -----------------------------
def refine_json(text, case_style, remove_nulls, pretty_print):
    try:
        data = json.loads(text)

        # Remove null values
        if remove_nulls:
            data = {k: v for k, v in data.items() if v is not None}

        # Case conversion
        if case_style == "UPPER":
            data = {k.upper(): v for k, v in data.items()}
        elif case_style == "lower":
            data = {k.lower(): v for k, v in data.items()}

        # Pretty print
        if pretty_print:
            output = json.dumps(data, indent=4)
        else:
            output = json.dumps(data)

        status = "✅ JSON refined successfully"
        stats = f"Total keys: {len(data)}"

        return output, status, stats

    except Exception as e:
        return "", f"❌ Error: {str(e)}", ""


# -----------------------------
# JSON Validation Function
# -----------------------------
def validate_json(text, required_fields):
    try:
        data = json.loads(text)

        required = [f.strip() for f in required_fields.split(",")]

        missing = [f for f in required if f not in data]

        if missing:
            return f"❌ Missing fields: {missing}"
        else:
            return "✅ JSON is valid"

    except Exception as e:
        return f"❌ Invalid JSON: {str(e)}"


# -----------------------------
# UI Layout
# -----------------------------
with gr.Blocks(title="JSON Toolkit") as demo:

    gr.Markdown("# 🧰 JSON Toolkit")

    with gr.Tabs():

        # =====================
        # TAB 1 : JSON Refiner
        # =====================
        with gr.Tab("✨ JSON Refiner"):

            with gr.Row():

                with gr.Column():

                    text_input = gr.Code(
                        language="json",
                        label="Input JSON",
                        lines=12,
                        value='{"name":"John","age":28,"email":null}'
                    )

                    case_style = gr.Radio(
                        ["None", "UPPER", "lower"],
                        value="None",
                        label="Case Style"
                    )

                    remove_nulls = gr.Checkbox(
                        label="Remove Null Values"
                    )

                    pretty_print = gr.Checkbox(
                        label="Pretty Print JSON",
                        value=True
                    )

                    refine_btn = gr.Button("✨ Refine JSON", variant="primary")

                with gr.Column():

                    json_output = gr.Code(
                        language="json",
                        label="Refined JSON",
                        lines=12
                    )

                    status_text = gr.Markdown()
                    stats_text = gr.Markdown()

            refine_btn.click(
                refine_json,
                inputs=[text_input, case_style, remove_nulls, pretty_print],
                outputs=[json_output, status_text, stats_text]
            )

        # =====================
        # TAB 2 : JSON Validation
        # =====================
        with gr.Tab("✅ Validation"):

            with gr.Row():

                with gr.Column():

                    json_to_validate = gr.Code(
                        language="json",
                        label="JSON to Validate",
                        lines=10,
                        value='{"name":"John","age":28}'
                    )

                    required_fields_input = gr.Textbox(
                        label="Required Fields (comma-separated)",
                        value="name, age, email"
                    )

                    validate_btn = gr.Button("🔍 Validate")

                with gr.Column():

                    validation_output = gr.Markdown()

            validate_btn.click(
                validate_json,
                inputs=[json_to_validate, required_fields_input],
                outputs=validation_output
            )


demo.launch()

In [ ]:
import json

def validate_json(json_text, schema_text, required_fields):
    try:
        data = json.loads(json_text)

        results = []

        # Required fields check
        if required_fields:
            fields = [f.strip() for f in required_fields.split(",")]
            missing = [f for f in fields if f not in data]

            if missing:
                results.append(f"❌ Missing fields: {missing}")
            else:
                results.append("✅ All required fields present")

        # Schema validation (simple type check)
        if schema_text:
            schema = json.loads(schema_text)

            for key, value_type in schema.items():
                if key in data:
                    if value_type == "string" and not isinstance(data[key], str):
                        results.append(f"❌ {key} should be string")

                    if value_type == "number" and not isinstance(data[key], (int, float)):
                        results.append(f"❌ {key} should be number")

        status = "✅ Validation completed"

        return "\n".join(results), status

    except Exception as e:
        return f"❌ Error: {str(e)}", "❌ Validation failed"

In [ ]:
import json
import re

def convert_key_case(data, target_case):
    def to_snake(s):
        return re.sub(r'(?<!^)(?=[A-Z])', '_', s).lower()

    def to_camel(s):
        parts = re.split(r'[_\- ]+', s)
        return parts[0].lower() + ''.join(p.title() for p in parts[1:])

    def to_kebab(s):
        return re.sub(r'(?<!^)(?=[A-Z])', '-', s).lower()

    def to_pascal(s):
        parts = re.split(r'[_\- ]+', s)
        return ''.join(p.title() for p in parts)

    converters = {
        "snake_case": to_snake,
        "camelCase": to_camel,
        "kebab-case": to_kebab,
        "PascalCase": to_pascal
    }

    convert = converters.get(target_case, lambda x: x)

    if isinstance(data, dict):
        return {convert(k): convert_key_case(v, target_case) for k, v in data.items()}
    elif isinstance(data, list):
        return [convert_key_case(i, target_case) for i in data]
    else:
        return data


def convert_json_case(json_text, target_case):
    try:
        data = json.loads(json_text)
        converted = convert_key_case(data, target_case)

        return json.dumps(converted, indent=4), "✅ Case conversion successful"

    except Exception as e:
        return "", f"❌ Error: {str(e)}"

In [ ]:
!pip install gradio==4.36.1

In [ ]:
import gradio as gr

def hello(name):
    return f"Hello {name}!"

with gr.Blocks() as demo:
    gr.Markdown("# Working Test")

    name = gr.Textbox()
    output = gr.Textbox()

    btn = gr.Button("Run")
    btn.click(hello, name, output)

demo.launch(share=True)

In [1]:
import gradio as gr
import json

# -------- Functions --------

def refine_json(text):
    try:
        data = {}
        lines = text.split("\n")

        for line in lines:
            if ":" in line:
                key, value = line.split(":", 1)
                data[key.strip()] = value.strip()

        return json.dumps(data, indent=4), "✅ JSON created"

    except Exception as e:
        return "", f"❌ Error: {e}"


def validate_json(json_text):
    try:
        json.loads(json_text)
        return "✅ JSON is valid"
    except Exception as e:
        return f"❌ Invalid JSON: {e}"


def convert_case(json_text):
    try:
        data = json.loads(json_text)
        new_data = {k.lower(): v for k, v in data.items()}
        return json.dumps(new_data, indent=4)
    except:
        return "Invalid JSON"


# -------- UI --------

with gr.Blocks(title="JSON Refiner") as demo:

    gr.Markdown("# JSON Refiner Tool")

    with gr.Tabs():

        # -------- TAB 1 --------
        with gr.Tab("Core Refining"):

            input_text = gr.Textbox(
                label="Key Value Input",
                lines=10,
                value="name: John\nage: 28\ncity: London"
            )

            refine_btn = gr.Button("Refine")

            output_json = gr.Code(language="json")

            status = gr.Markdown()

            refine_btn.click(
                refine_json,
                inputs=input_text,
                outputs=[output_json, status]
            )

        # -------- TAB 2 --------
        with gr.Tab("Validation"):

            json_input = gr.Code(
                language="json",
                value='{"name":"John"}'
            )

            validate_btn = gr.Button("Validate")

            validation_result = gr.Markdown()

            validate_btn.click(
                validate_json,
                inputs=json_input,
                outputs=validation_result
            )

        # -------- TAB 3 --------
        with gr.Tab("Case Conversion"):

            case_input = gr.Code(
                language="json",
                value='{"Name":"John","Age":28}'
            )

            convert_btn = gr.Button("Convert to lowercase keys")

            case_output = gr.Code(language="json")

            convert_btn.click(
                convert_case,
                inputs=case_input,
                outputs=case_output
            )

demo.launch(debug=False)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ae570ebc4961232610.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
